# CoA Instrument Mapping Probe → Excel Output

Reads the enriched `CoA_Instrument_Mapping_Probe.csv` and produces an Excel
workbook with a README sheet and a formatted data sheet.

**Color convention across the repo:**
- **Green** (`#548235`) — individual test code / TESTCD identity (green track)
- **Yellow** (`#FFD700`) — COSMoS dataset specialisation layer
- **Dark chocolate** (`#5B3A29`) — instrument identity (this probe, parent level)
- **Copper** (`#CC7A3E`) — question container / subscale identity (this probe, middle level)

The two browns signal that this file resolves *two* hierarchy levels above green:
the instrument and the container (subscale). Dark chocolate vs copper makes the
two levels unmistakable at a glance.

## Setup


In [ ]:
import os
import pandas as pd
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from datetime import datetime

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
PROBE_CSV = os.path.join(REPO_ROOT, 'interim', 'CoA_Instrument_Mapping_Probe.csv')
OUTPUT_XLSX = os.path.join(REPO_ROOT, 'interim', 'CoA_Instrument_Mapping_Probe.xlsx')

assert os.path.exists(PROBE_CSV), f'Probe CSV not found: {PROBE_CSV}'
print(f'Input:  {PROBE_CSV}')
print(f'Output: {OUTPUT_XLSX}')


## Color palette

Two brown-family tones with maximum contrast for the two hierarchy levels:
- **Dark chocolate** (`#5B3A29`) — instrument (the parent assessment)
- **Copper** (`#CC7A3E`) — question container / subscale (the middle grouping)

Consistent with the repo convention where green = TESTCD identity
and yellow = COSMoS specialisations.

In [ ]:
# ── Color palette ──
# Instrument level (parent assessment) — dark chocolate brown
INSTRUMENT_HEADER = PatternFill('solid', fgColor='5B3A29')   # dark chocolate – instrument headers
INSTRUMENT_LIGHT  = PatternFill('solid', fgColor='EDE0D4')   # warm cream – instrument alt rows

# Container / subscale level (middle grouping) — distinct copper
CONTAINER_HEADER  = PatternFill('solid', fgColor='CC7A3E')   # copper – container headers
CONTAINER_LIGHT   = PatternFill('solid', fgColor='FAEADC')   # light peach – container alt rows

# Neutral (codelist key columns + flags)
NEUTRAL_HEADER    = PatternFill('solid', fgColor='7F7F7F')   # medium grey – key/flag columns
NEUTRAL_LIGHT     = PatternFill('solid', fgColor='F2F2F2')   # light grey – key/flag alt rows

# README
BROWN_SECTION     = PatternFill('solid', fgColor='D4A574')   # warm tan – README section headers

WHITE_BOLD   = Font(name='Arial', bold=True, size=10, color='FFFFFF')
TITLE_FONT   = Font(name='Arial', bold=True, size=12, color='4A2F1A')
SECTION_FONT = Font(name='Arial', bold=True, size=10, color='4A2F1A')
BODY_FONT    = Font(name='Arial', size=10)
THIN_BORDER  = Border(bottom=Side(style='thin', color='D9D9D9'))

# ── Column → level mapping ──
# Determines header color and alternating-row tint per column
INSTRUMENT_COLS = {
    'instrument_ncit_code', 'instrument_ncit_name', 'match_method',
    'instrument_definition', 'instrument_synonyms', 'instrument_display_name',
    'instrument_UMLS_CUI', 'instrument_NCIm_CUI',
}
CONTAINER_COLS = {
    'container_ncit_code', 'container_name',
    'container_definition', 'container_synonyms',
    'container_UMLS_CUI', 'container_NCIm_CUI',
}
# Everything else (codelist_testcd, instrument_label, question_count,
# has_container, has_instrument) gets NEUTRAL

def col_fills(col_name):
    """Return (header_fill, alt_row_fill) for a column."""
    if col_name in INSTRUMENT_COLS:
        return INSTRUMENT_HEADER, INSTRUMENT_LIGHT
    elif col_name in CONTAINER_COLS:
        return CONTAINER_HEADER, CONTAINER_LIGHT
    else:
        return NEUTRAL_HEADER, NEUTRAL_LIGHT

## Load and reshape probe data

Drop `instrument_semantic_type` and `container_semantic_type` (always
"Intellectual Product" — no discriminating value).

In [ ]:
df = pd.read_csv(PROBE_CSV)
print(f'Raw: {len(df)} rows, {len(df.columns)} columns')

# ── Drop semantic_type columns (always "Intellectual Product") ──
df = df.drop(columns=['instrument_semantic_type', 'container_semantic_type'])

print(f'Final: {len(df)} rows, {len(df.columns)} columns')
print(f'Columns: {list(df.columns)}')

## Build README sheet

Follows the repo convention: provenance, purpose, scope, column descriptions,
design decisions. Brown section headers instead of default grey.


In [ ]:
wb = openpyxl.Workbook()
ws = wb.active
ws.title = 'README'

def add_row(ws, row, text, font=BODY_FONT, fill=None):
    c = ws.cell(row=row, column=1, value=text)
    c.font = font
    c.alignment = Alignment(wrap_text=True, vertical='top')
    if fill:
        c.fill = fill
    return row + 1

r = 1
r = add_row(ws, r, 'CoA INSTRUMENT MAPPING PROBE — NCIt ENRICHED REFERENCE FILE', TITLE_FONT)
r += 1

# ── Provenance ──
r = add_row(ws, r, 'PROVENANCE', SECTION_FONT, BROWN_SECTION)
r = add_row(ws, r, f'Generated:      {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
r = add_row(ws, r, 'Pipeline:       CoA_Instrument_Identity_Probe.ipynb → CoA_Probe_to_Excel.ipynb')
r = add_row(ws, r, '                (explore/coa-structure branch)')
r = add_row(ws, r, 'SDTM CT source: NCI EVS SDTM Terminology 2026-03-27')
r = add_row(ws, r, 'NCIt source:    Thesaurus.FLAT.zip (NCI EVS)')
r = add_row(ws, r, 'CUI source:     nci_code_cui_map.dat (NCI EVS, updated on NCIm release schedule)')
r = add_row(ws, r, 'COSMoS ref:     BC/DSS package 2026-Q1 (for context, not used as input)')
r += 1

# ── Purpose ──
r = add_row(ws, r, 'PURPOSE', SECTION_FONT, BROWN_SECTION)
r = add_row(ws, r, 'Maps SDTM instrument TESTCD codelists to a three-level NCIt hierarchy:')
r = add_row(ws, r, '  Instrument (C20993 descendants) → Question Container (C211913 descendants) → TESTCD Codelist')
r = add_row(ws, r, 'Each row represents one TESTCD codelist and its resolved instrument and container parents.')
r = add_row(ws, r, 'Enriched with NCIt identity (definitions, synonyms, UMLS CUIs) at instrument and container levels.')
r += 1

# ── Scope (computed) ──
r = add_row(ws, r, 'SCOPE', SECTION_FONT, BROWN_SECTION)
n_instr = df['instrument_ncit_code'].nunique()
n_cont = df[df['has_container'] == True]['container_ncit_code'].nunique()
n_has_instr = int(df['has_instrument'].sum())
n_has_cont = int(df['has_container'].sum())
n_umls_i = int(df['instrument_UMLS_CUI'].notna().sum())
n_umls_c = int(df['container_UMLS_CUI'].notna().sum())
r = add_row(ws, r, f'{len(df)} TESTCD codelists across QS, FT, RS domains.')
r = add_row(ws, r, f'{n_instr} distinct instruments resolved (has_instrument=True: {n_has_instr}).')
r = add_row(ws, r, f'{n_cont} distinct question containers resolved (has_container=True: {n_has_cont}).')
r = add_row(ws, r, f'Instrument-level UMLS CUI: {n_umls_i} ({n_umls_i/len(df)*100:.1f}%).')
r = add_row(ws, r, f'Container-level UMLS CUI: {n_umls_c} ({n_umls_c/len(df)*100:.1f}%).')
r += 1


## Column descriptions


In [ ]:
COL_DOCS = [
    ('codelist_testcd',         'SDTM TESTCD codelist submission value (e.g. SIXMW1TC) — join key to green track'),
    ('instrument_label',        'Instrument name derived from codelist label'),
    ('question_count',          'Number of individual test codes in this codelist'),
    ('instrument_ncit_code',    'NCIt code for the parent Instrument concept (C20993 descendant)'),
    ('instrument_ncit_name',    'NCIt preferred name for the instrument'),
    ('match_method',            'How the instrument was resolved: exact, label_match, or parent_walk'),
    ('instrument_definition',   'NCIt definition of the instrument concept'),
    ('instrument_synonyms',     'All NCIt synonyms for the instrument (pipe-separated, undifferentiated)'),
    ('instrument_display_name', 'NCIt display name for the instrument'),
    ('instrument_UMLS_CUI',     'Standard UMLS CUI (C + digits) — crosslinkable to SNOMED, MeSH'),
    ('instrument_NCIm_CUI',     'NCI Metathesaurus CUI (CL + digits) — NCI-internal'),
    ('container_ncit_code',     'NCIt code for the Question Container (C211913 descendant)'),
    ('container_name',          'NCIt preferred name for the question container'),
    ('container_definition',    'NCIt definition of the container concept'),
    ('container_synonyms',      'All NCIt synonyms for the container (pipe-separated)'),
    ('container_UMLS_CUI',      'Standard UMLS CUI for the container'),
    ('container_NCIm_CUI',      'NCI Metathesaurus CUI for the container'),
    ('has_container',           'Boolean: True if a question container was resolved'),
    ('has_instrument',          'Boolean: True if a parent instrument was resolved'),
]

r = add_row(ws, r, 'COLUMN DESCRIPTIONS', SECTION_FONT, BROWN_SECTION)
for col_name, col_desc in COL_DOCS:
    r = add_row(ws, r, f'{col_name:30s} {col_desc}')
r += 1

## Design decisions


In [ ]:
r = add_row(ws, r, 'DESIGN DECISIONS', SECTION_FONT, BROWN_SECTION)
decisions = [
    '1. Hierarchy probe uses NCIt parent-child relationships from Thesaurus.FLAT, not COSMoS metadata.',
    '2. Synonym enrichment is undifferentiated (NCI/CDISC source tags not available in FLAT file).',
    '3. Pipe separator (|) for synonyms matches NCIt FLAT convention; semicolons reserved for COSMoS Categories.',
    '4. UMLS vs NCIm CUI split: standard CUIs (C+digits) are interoperable; CL-prefixed are NCI-internal.',
    '5. Container level corresponds to subscale/section grouping within an instrument.',
    '6. Instrument synonyms enable derivation of COSMoS Category tokens (e.g. 6MWT, SIXMW1 from C115789 synonyms).',
]
for d in decisions:
    r = add_row(ws, r, d)
r += 1

r = add_row(ws, r, 'COLOR CONVENTION', SECTION_FONT, BROWN_SECTION)
r = add_row(ws, r, 'Dark chocolate (#5B3A29) — instrument identity columns (parent level)')
r = add_row(ws, r, 'Copper (#CC7A3E)         — question container / subscale identity columns (middle level)')
r = add_row(ws, r, 'Grey (#7F7F7F)           — codelist key columns and resolution flags')
r = add_row(ws, r, 'Green (#548235)          — individual test code / TESTCD identity (green track, other files)')
r = add_row(ws, r, 'Yellow (#FFD700)         — COSMoS dataset specialisation layer (other files)')

ws.column_dimensions['A'].width = 120
print(f'README: {r-1} rows')

## Build data sheet

Per-column color coding: dark brown for instrument columns, sienna for container
columns, grey for the codelist key and resolution flags. Alternating rows use
the light tint of each column's level color.

In [ ]:
ws_data = wb.create_sheet('Instrument Mapping Probe')

# ── Column widths ──
COL_WIDTHS = {
    'codelist_testcd': 16, 'instrument_label': 45, 'question_count': 14,
    'instrument_ncit_code': 18, 'instrument_ncit_name': 45, 'match_method': 14,
    'instrument_definition': 55, 'instrument_synonyms': 50,
    'instrument_display_name': 30,
    'instrument_UMLS_CUI': 16, 'instrument_NCIm_CUI': 16,
    'container_ncit_code': 18, 'container_name': 40, 'container_definition': 50,
    'container_synonyms': 40, 'container_UMLS_CUI': 16, 'container_NCIm_CUI': 16,
    'has_container': 14, 'has_instrument': 14,
}

# ── Pre-compute fills per column index (1-based) ──
col_header_fills = {}
col_alt_fills = {}
for c_idx, col_name in enumerate(df.columns, 1):
    hdr, alt = col_fills(col_name)
    col_header_fills[c_idx] = hdr
    col_alt_fills[c_idx] = alt

# ── Header row ──
for c_idx, col_name in enumerate(df.columns, 1):
    cell = ws_data.cell(row=1, column=c_idx, value=col_name)
    cell.font = WHITE_BOLD
    cell.fill = col_header_fills[c_idx]
    cell.alignment = Alignment(horizontal='center', wrap_text=True)
    ws_data.column_dimensions[get_column_letter(c_idx)].width = COL_WIDTHS.get(col_name, 15)

# ── Data rows with per-column alternating fill ──
for r_idx, row in enumerate(df.itertuples(index=False), 2):
    is_alt = (r_idx % 2 == 0)
    for c_idx, val in enumerate(row, 1):
        cell = ws_data.cell(row=r_idx, column=c_idx)
        cell.value = val if pd.notna(val) else None
        cell.font = BODY_FONT
        cell.alignment = Alignment(vertical='top')
        cell.border = THIN_BORDER
        if is_alt:
            cell.fill = col_alt_fills[c_idx]

# ── Autofilter + freeze ──
ws_data.auto_filter.ref = f'A1:{get_column_letter(len(df.columns))}{len(df)+1}'
ws_data.freeze_panes = 'A2'

print(f'Data sheet: {len(df)} rows × {len(df.columns)} columns')
print(f'Header colors: instrument={sum(1 for f in col_header_fills.values() if f == INSTRUMENT_HEADER)} cols, '
      f'container={sum(1 for f in col_header_fills.values() if f == CONTAINER_HEADER)} cols, '
      f'neutral={sum(1 for f in col_header_fills.values() if f == NEUTRAL_HEADER)} cols')

## Save workbook


In [ ]:
wb.save(OUTPUT_XLSX)
print(f'Saved: {OUTPUT_XLSX}')
print(f'Sheets: {wb.sheetnames}')
print(f'Size: {os.path.getsize(OUTPUT_XLSX):,} bytes')


## Verification


In [ ]:
# Re-open and spot-check
wb2 = openpyxl.load_workbook(OUTPUT_XLSX)
ws2 = wb2['Instrument Mapping Probe']
print(f'Verified: {ws2.max_row} rows × {ws2.max_column} columns')

# Check header colors across column groups
print('\nHeader colors by column:')
for c in range(1, ws2.max_column + 1):
    name = ws2.cell(1, c).value
    rgb = ws2.cell(1, c).fill.fgColor.rgb if ws2.cell(1, c).fill.fgColor else 'none'
    if '5B3A29' in rgb:
        level = 'INSTRUMENT (dark chocolate)'
    elif 'CC7A3E' in rgb:
        level = 'CONTAINER (copper)'
    elif '7F7F7F' in rgb:
        level = 'NEUTRAL (grey)'
    else:
        level = f'UNKNOWN ({rgb})'
    print(f'  {name:30s} {rgb}  {level}')

# Spot-check 6MWT — find by codelist_testcd (col 1)
print()
for r in range(2, ws2.max_row + 1):
    if ws2.cell(r, 1).value == 'SIXMW1TC':
        print(f'6MWT (row {r}):')
        for c in range(1, ws2.max_column + 1):
            print(f'  {ws2.cell(1,c).value}: {ws2.cell(r,c).value}')
        break

ws_r = wb2['README']
print(f'\nREADME: {ws_r.max_row} rows')